In [0]:
reviews = spark.read.table("revenue_operations.bronze.reviews")
geolocation = spark.read.table("revenue_operations.bronze.geolocation")
product_category_translation = spark.read.table("revenue_operations.bronze.product_category_translation")
products = spark.read.table("revenue_operations.bronze.products")
orders = spark.read.table("revenue_operations.bronze.orders")
sellers = spark.read.table("revenue_operations.bronze.sellers")
customers = spark.read.table("revenue_operations.bronze.customers")
order_items = spark.read.table("revenue_operations.bronze.order_items")

### Silver reviews

In [0]:
reviews.show(5)
reviews.count()
reviews.printSchema()

In [0]:
from pyspark.sql.functions import column, lit
# Check for duplicate rows
duplicate_reviews = reviews.count() - reviews.dropDuplicates().count()
print(f"Duplicate rows count in reviews : {duplicate_reviews}")

# Checking for null values
for col in reviews.columns:
    print(f"Null value in {col}: {reviews.filter(column(col).isNull()).count()}")

print("\n")
# Checking for distinct values
for col in reviews.columns:
    print(f"Number of distinct values in {col}: {reviews.select(col).distinct().count()}")
    print(f"Number of duplicate values in {col}: {reviews.select(col).count() - reviews.select(col).dropDuplicates().count()}")

print("\n")
# review_id and order_id combined are the composite key, checking for combined duplicates
print(f"Number of duplicates in composite key: {reviews.select('review_id','order_id').count() - reviews.select('review_id','order_id').dropDuplicates().count()}")

In [0]:
# Foreign Key check: Do any reviews reference order_ids that don't exist in Orders?
invalid_review_order_ids = reviews.join(orders.select("order_id"), on="order_id", how="leftanti").count()
print(f"Reviews with invalid order_ids (not in Orders table): {invalid_review_order_ids}")

Reviews Silver Design Notes

Source = revenue_operations.bronze.reviews

Target = revenue_operations.silver.reviews

Columns renamed = None

Data type Changes = None

Duplicate findings = No duplicate rows. Duplicates in individual columns (scores, titles, messages, timestamps) are expected. CRITICAL DATA QUALITY ISSUE: 789 review_ids map to 2-3 different customers each (ID collision in source system). The composite key (review_id, order_id) IS unique and will serve as the primary key. 

Null findings = Multiple Null values across the comment title and message.

Primary Key = review_id + order_id

Validation findings = None

Foreign Key findings = All reviews.order_id exist in Orders table (0 invalid references)

Audit columns = source_file_name, ingestion_timestamp, silver_processed_timestamp, data_quality_status (values: 'valid' for clean records, 'duplicate_review_id_across_customers' for 789 records with ID collision, 'missing_parent_order_id' for reviews with no matching order)

In [0]:
# Test hypothesis: duplicate review_ids = same customer reviewing multiple orders

# Join reviews → orders → customers to link review_id to customer_id
reviews_with_customer = reviews.join(orders.select("order_id", "customer_id"), on="order_id", how="inner")

# Find review_ids that appear multiple times
from pyspark.sql.functions import count as spark_count
review_id_counts = reviews_with_customer.groupBy("review_id").agg(spark_count("*").alias("review_count"))
duplicate_review_ids = review_id_counts.filter("review_count > 1")

print(f"Number of review_ids with duplicates: {duplicate_review_ids.count()}")
print(f"\nSample of duplicate review_ids:")
duplicate_review_ids.orderBy("review_count", ascending=False).show(10)

# For duplicate review_ids, check if they map to the same customer
print("\n" + "="*80)
print("Testing: Does each duplicate review_id belong to ONE customer or MULTIPLE?")
print("="*80)

duplicate_review_details = reviews_with_customer.join(
    duplicate_review_ids.select("review_id"), 
    on="review_id", 
    how="inner"
)

# Group by review_id and count distinct customers per review_id
from pyspark.sql.functions import countDistinct
customers_per_review = duplicate_review_details.groupBy("review_id").agg(
    countDistinct("customer_id").alias("distinct_customers"),
    spark_count("order_id").alias("total_orders")
)

print("\nDo duplicate review_ids map to ONE customer (hypothesis TRUE) or MULTIPLE (data error)?")
customers_per_review.groupBy("distinct_customers").count().orderBy("distinct_customers").show()

# Show examples
print("\nExample: Same review_id used across different orders:")
example_review_id = duplicate_review_ids.first()["review_id"]
print(f"\nShowing all orders for review_id: {example_review_id}")
reviews_with_customer.filter(f"review_id = '{example_review_id}'").select(
    "review_id", "order_id", "customer_id", "review_score", "review_creation_date"
).show(truncate=False)

### Silver Reviews Dataframe

In [0]:
silver_reviews = reviews.select("*")

from pyspark.sql.functions import current_timestamp, lit, when, col, count

print(f"Before : {silver_reviews.count()}")
silver_reviews = silver_reviews.dropDuplicates()
print(f"After : {silver_reviews.count()}")

duplicate_review_ids = reviews.groupBy("review_id").agg(count("*").alias("count")).filter("count > 1")
duplicate_review_ids_list = [row.review_id for row in duplicate_review_ids.collect()]

silver_reviews = (silver_reviews
    .withColumn("source_file_name", lit("olist_orders_reviews.csv"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("silver_processed_timestamp", current_timestamp())
    .withColumn("data_quality_status", 
        when(col("review_id").isin(duplicate_review_ids_list), lit("duplicate_review_id_across_customers"))
        .when(col("review_id").isNull() | col("order_id").isNull(), lit("missing_key"))
        .when(
            col("review_score").isNull() |  
            col("review_comment_title").isNull() | 
            col("review_comment_message").isNull() | 
            col("review_creation_date").isNull() | 
            col("review_answer_timestamp").isNull(), 
            lit("missing_value")
        )
        .otherwise(lit("valid"))
    )
)

display(silver_reviews.limit(5))

In [0]:
silver_reviews.write.format("delta").mode("overwrite").saveAsTable("revenue_operations.silver.reviews")

In [0]:
reviews_silver = spark.read.table("revenue_operations.silver.reviews")
reviews_silver.show()
reviews_silver.printSchema()

### Silver Product Category Translation

In [0]:
product_category_translation.show(5)
product_category_translation.printSchema()

In [0]:
print("Total number of product category names: ", product_category_translation.count())
# Duplicates count
print("Number of duplicate rows: ", product_category_translation.count() - product_category_translation.dropDuplicates().count())

# Null Values
from pyspark.sql.functions import col

for col_name in product_category_translation.columns:
    print(f"Number of null values in {col_name}:  {product_category_translation.filter(col(col_name).isNull()).count()}")

# Distinct Values
for col_name in product_category_translation.columns:
    print(f"Distinct Values in {col_name} : {product_category_translation.select(col_name).distinct().count()}")


In [0]:
# Foreign Key check: Do any products reference categories without translations?
products_missing_translation = products.join(product_category_translation, on="product_category_name", how="leftanti").count()
print(f"Products with categories lacking translation: {products_missing_translation}")
print("\nNote: These products will be kept in Silver. Translation is optional enrichment.")


Product Category Translation Silver Design Notes

Source = revenue_operations.bronze.product_category_translation

Target = revenue_operations.silver.product_category_translation

Columns renamed = None

Data type Changes = None

Duplicate findings = No duplicate rows.

Null findings = None

Primary Key = product_category_name

Validation findings: None

Foreign Key findings = 623 products have categories without translation (kept in Silver; translation is optional enrichment)

Audit columns = source_file_name, ingestion_timestamp, silver_processed_timestamp, data_quality_status (values: 'valid' for clean records)

In [0]:
silver_product_category_translation = product_category_translation.select("*")

# No duplicates and null values/ adding audit columns
silver_product_category_translation = silver_product_category_translation.withColumn("source_file_name", lit("product_category_name_translation.csv")).withColumn("ingestion_timestamp", current_timestamp()).withColumn("silver_processed_timestamp", current_timestamp()).withColumn("data_quality_status", lit("valid"))

display(silver_product_category_translation.limit(5))


In [0]:
silver_product_category_translation.write.format("delta").mode("overwrite").saveAsTable("revenue_operations.silver.product_category_translation")
product_category_translation_silver = spark.read.table("revenue_operations.silver.product_category_translation")
product_category_translation_silver.limit(10).display()

In [0]:
product_category_translation_silver.printSchema()

### Silver Geolocation

In [0]:
display(geolocation.limit(5))
geolocation.printSchema()

   
Geolocation Silver Design Notes

Source = revenue_operations.bronze.geolocation

Target = revenue_operations.silver.geolocation

Columns renamed = geolocation_lat → geolocation_avg_latitude, geolocation_lng → geolocation_avg_longitude

Data type Changes = None

Duplicate findings = Bronze table has 1,000,163 rows with 19,015 unique ZIP codes. Many ZIPs have multiple (city, state) combinations.

Transformation Logic:
1. For each ZIP code, selected the most frequent (city, state) combination using window function
2. Averaged latitude/longitude for the selected (zip, city, state) combination

Null findings = None

Primary Key = geolocation_zip_code_prefix

Validation findings:
- Final row count: 19,015 (one per unique ZIP)
- No duplicate ZIPs
- 157 Customer ZIPs have no geolocation match (will result in NULL joins)
- 7 Seller ZIPs have no geolocation match (will result in NULL joins)

Foreign Key findings = Not applicable (this is a dimension table)

Audit columns = source_file_name, ingestion_timestamp, silver_processed_timestamp

In [0]:
geolocation.groupBy('geolocation_zip_code_prefix', 'geolocation_city', 'geolocation_state').count().orderBy("count", ascending = False).show(5)


In [0]:
geolocation.select("geolocation_zip_code_prefix").distinct().count()
geolocation.filter(column("geolocation_zip_code_prefix") == 24220).show()

For zip codes to be primary key:
 - Average all the latitue and longitude that is in those row of the zip code.
 - Take the most occuring city in that zip code.
 - Take the state in that zip code (usually state will not be different in a single zip code).

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, count, avg, first

# Unique Zip Codes
silver_geolocation = geolocation.select("geolocation_zip_code_prefix").distinct()
print("Unique Zip codes in Silver geolocation dataframe", silver_geolocation.count())
print("Unique Zip codes in Bronze geolocation dataframe", geolocation.select("geolocation_zip_code_prefix").distinct().count())

# Most Occurring City - select the most frequent (city, state) combination per ZIP
city_counts = geolocation.groupBy('geolocation_zip_code_prefix', 'geolocation_city', 'geolocation_state').agg(count('*').alias("city_occurence_count"))

ranked_city = city_counts.withColumn("rank", row_number().over(Window.partitionBy("geolocation_zip_code_prefix").orderBy(col("city_occurence_count").desc())))

silver_geolocation = ranked_city.select('*').filter(column("rank") == 1)
silver_geolocation.show()
print("Unique Zip codes in Silver geolocation dataframe", silver_geolocation.count())

# Averages of longitude and latitudes
avg_coordinates = geolocation.groupBy("geolocation_zip_code_prefix", "geolocation_city","geolocation_state").agg(
    avg("geolocation_lat").alias("geolocation_avg_latitude"),
    avg("geolocation_lng").alias("geolocation_avg_longitude")
)
silver_geolocation = silver_geolocation.join(avg_coordinates, on = ["geolocation_zip_code_prefix", "geolocation_city", "geolocation_state"], how = "inner")
silver_geolocation.sort('geolocation_zip_code_prefix').show()

print("Unique Zip codes in Silver geolocation dataframe", silver_geolocation.count())

# Add audit columns
silver_geolocation = silver_geolocation.withColumn("source_file_name", lit("olist_geolocation_dataset.csv"))\
    .withColumn("ingestion_timestamp", current_timestamp())\
    .withColumn("silver_processed_timestamp", current_timestamp())

# Drop intermediate columns
silver_geolocation = silver_geolocation.drop("city_occurence_count", "rank")
display(silver_geolocation.show())

In [0]:
print(f"Final row count: {silver_geolocation.count()}")

print(f"Distinct ZIP codes: { silver_geolocation.select("geolocation_zip_code_prefix").distinct().count()}")

print(f"Duplicate Zip Codes: {silver_geolocation.groupBy("geolocation_zip_code_prefix").count().filter(col("count") > 1).count()}")


In [0]:
# Checking if all the customers and sellers have zip codes matching in the new silver geolocation table
customer_missing_zip = customers.join(
    silver_geolocation, customers.customer_zip_code_prefix == silver_geolocation.geolocation_zip_code_prefix, how="left"
).filter(col("geolocation_zip_code_prefix").isNull()).count()
print("Customer zip codes not matching with silver geolocation =", customer_missing_zip)

seller_missing_zip = sellers.join(
    silver_geolocation, sellers.seller_zip_code_prefix == silver_geolocation.geolocation_zip_code_prefix, how="left"
).filter(col("geolocation_zip_code_prefix").isNull()).count()
print("Seller zip codes not matching with silver geolocation =", seller_missing_zip)


In [0]:
silver_geolocation.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("revenue_operations.silver.geolocation")
geolocation_silver = spark.read.table("revenue_operations.silver.geolocation")
display(geolocation_silver.limit(5))
geolocation_silver.printSchema()